In [1]:
%%writefile pipeline/monitoring/drift_detector.py
"""
Drift Detection Service
Enterprise MLOps Platform
"""

import numpy as np
import pandas as pd
from scipy import stats
from datetime import datetime


class DriftDetector:
    """
    Monitors incoming predictions for data drift.
    Compares feature distributions against training data baseline.
    """
    
    def __init__(self, reference_data: pd.DataFrame, psi_threshold: float = 0.2, ks_threshold: float = 0.05):
        self.reference = reference_data
        self.psi_threshold = psi_threshold
        self.ks_threshold = ks_threshold
        self.reference_stats = self._compute_stats(reference_data)
        self.alerts = []
    
    def _compute_stats(self, data: pd.DataFrame) -> dict:
        return {
            col: {"mean": data[col].mean(), "std": data[col].std(),
                   "min": data[col].min(), "max": data[col].max()}
            for col in data.columns
        }
    
    def calculate_psi(self, expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
        """
        Population Stability Index.
        PSI < 0.1: no significant shift
        PSI 0.1-0.2: moderate shift, monitor closely
        PSI > 0.2: significant shift, investigate/retrain
        """
        breakpoints = np.linspace(
            min(expected.min(), actual.min()),
            max(expected.max(), actual.max()),
            bins + 1
        )
        
        expected_counts = np.histogram(expected, bins=breakpoints)[0] + 1
        actual_counts = np.histogram(actual, bins=breakpoints)[0] + 1
        
        expected_pct = expected_counts / expected_counts.sum()
        actual_pct = actual_counts / actual_counts.sum()
        
        psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
        return float(psi)
    
    def check_drift(self, current_data: pd.DataFrame) -> dict:
        """Run drift checks on incoming data against reference baseline."""
        results = {
            "timestamp": datetime.now().isoformat(),
            "samples_checked": len(current_data),
            "features_drifted": [],
            "overall_status": "healthy",
            "details": {}
        }
        
        for col in self.reference.columns:
            if col not in current_data.columns:
                continue
            
            ref_values = self.reference[col].dropna().values
            cur_values = current_data[col].dropna().values
            
            if len(cur_values) < 10:
                continue
            
            # PSI
            psi = self.calculate_psi(ref_values, cur_values)
            
            # KS test
            ks_stat, ks_pvalue = stats.ks_2samp(ref_values, cur_values)
            
            # Mean shift
            ref_mean = ref_values.mean()
            cur_mean = cur_values.mean()
            mean_shift = abs(cur_mean - ref_mean) / (ref_values.std() + 1e-6)
            
            is_drifted = psi > self.psi_threshold or ks_pvalue < self.ks_threshold
            
            results["details"][col] = {
                "psi": round(psi, 4),
                "ks_statistic": round(ks_stat, 4),
                "ks_pvalue": round(ks_pvalue, 4),
                "mean_shift_std": round(mean_shift, 4),
                "drifted": is_drifted
            }
            
            if is_drifted:
                results["features_drifted"].append(col)
        
        drift_count = len(results["features_drifted"])
        total_features = len(results["details"])
        drift_pct = drift_count / total_features if total_features > 0 else 0
        
        if drift_pct > 0.3:
            results["overall_status"] = "critical"
        elif drift_pct > 0.1:
            results["overall_status"] = "warning"
        
        if results["overall_status"] != "healthy":
            self.alerts.append({
                "timestamp": results["timestamp"],
                "status": results["overall_status"],
                "drifted_features": drift_count,
                "total_features": total_features
            })
        
        return results
    
    def get_alerts(self) -> list:
        return self.alerts


if __name__ == "__main__":
    np.random.seed(42)
    ref = pd.DataFrame(np.random.randn(1000, 5), columns=[f"f{i}" for i in range(5)])
    
    # No drift
    current_ok = pd.DataFrame(np.random.randn(200, 5), columns=[f"f{i}" for i in range(5)])
    
    # With drift — shift mean of 2 features
    current_drift = current_ok.copy()
    current_drift["f0"] = current_drift["f0"] + 2.0
    current_drift["f1"] = current_drift["f1"] * 3.0
    
    detector = DriftDetector(ref)
    
    print("No drift scenario:")
    r1 = detector.check_drift(current_ok)
    print(f"  Status: {r1['overall_status']}, Drifted: {len(r1['features_drifted'])}")
    
    print("\nDrift scenario:")
    r2 = detector.check_drift(current_drift)
    print(f"  Status: {r2['overall_status']}, Drifted: {len(r2['features_drifted'])}")
    for f in r2["features_drifted"]:
        print(f"  {f}: PSI={r2['details'][f]['psi']}, KS p={r2['details'][f]['ks_pvalue']}")

Writing pipeline/monitoring/drift_detector.py


In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from pipeline.ingestion.feature_engineering import engineer_features

# Load and engineer features
df = pd.read_csv('data/raw/creditcard.csv')
df_featured = engineer_features(df)

# Split: first 80% is our "training reference", last 20% is "production"
feature_cols = [col for col in df_featured.columns if col not in ['Class', 'Time']]
split = int(len(df_featured) * 0.8)

reference = df_featured[feature_cols].iloc[:split]
production = df_featured[feature_cols].iloc[split:]

from pipeline.monitoring.drift_detector import DriftDetector

# TEST 1: Real production data (should be healthy — same distribution)
detector = DriftDetector(reference)
result = detector.check_drift(production)

print("TEST 1: Real production data vs training reference")
print(f"  Status: {result['overall_status']}")
print(f"  Features drifted: {len(result['features_drifted'])} / {len(result['details'])}")
if result['features_drifted']:
    for f in result['features_drifted']:
        d = result['details'][f]
        print(f"    {f}: PSI={d['psi']}, KS p-value={d['ks_pvalue']}")

# TEST 2: Simulate drift — what happens when fraud patterns change
drifted = production.copy()
drifted['V14'] = drifted['V14'] + 2.0       # Shift a key fraud feature
drifted['V17'] = drifted['V17'] * 1.5       # Scale another
drifted['Amount'] = drifted['Amount'] * 3.0  # Transaction amounts triple

result2 = detector.check_drift(drifted)

print(f"\nTEST 2: Simulated drift (V14 shifted, V17 scaled, Amount tripled)")
print(f"  Status: {result2['overall_status']}")
print(f"  Features drifted: {len(result2['features_drifted'])} / {len(result2['details'])}")
for f in result2['features_drifted'][:10]:
    d = result2['details'][f]
    print(f"    {f}: PSI={d['psi']}, KS p-value={d['ks_pvalue']}")

print(f"\nAlerts raised: {len(detector.get_alerts())}")
for alert in detector.get_alerts():
    print(f"  [{alert['status'].upper()}] {alert['drifted_features']}/{alert['total_features']} features drifted")

TEST 1: Real production data vs training reference
  Status: critical
  Features drifted: 39 / 40
    V1: PSI=0.0009, KS p-value=0.0
    V2: PSI=0.0012, KS p-value=0.0
    V3: PSI=0.0533, KS p-value=0.0
    V4: PSI=0.0845, KS p-value=0.0
    V5: PSI=0.0006, KS p-value=0.0
    V6: PSI=0.0003, KS p-value=0.0
    V7: PSI=0.0011, KS p-value=0.0
    V8: PSI=0.001, KS p-value=0.0
    V9: PSI=0.0106, KS p-value=0.0
    V10: PSI=0.0052, KS p-value=0.0
    V11: PSI=0.1154, KS p-value=0.0
    V12: PSI=0.0834, KS p-value=0.0
    V13: PSI=0.0646, KS p-value=0.0
    V14: PSI=0.0435, KS p-value=0.0
    V15: PSI=0.1226, KS p-value=0.0
    V16: PSI=0.002, KS p-value=0.0
    V17: PSI=0.0074, KS p-value=0.0
    V18: PSI=0.0156, KS p-value=0.0
    V19: PSI=0.0118, KS p-value=0.0
    V20: PSI=0.0008, KS p-value=0.0
    V21: PSI=0.0006, KS p-value=0.0
    V22: PSI=0.0006, KS p-value=0.0
    V23: PSI=0.001, KS p-value=0.0
    V24: PSI=0.0253, KS p-value=0.0
    V25: PSI=0.0532, KS p-value=0.0
    V26: PSI=0

In [3]:
%%writefile pipeline/monitoring/drift_detector.py
"""
Drift Detection Service
Enterprise MLOps Platform
"""

import numpy as np
import pandas as pd
from scipy import stats
from datetime import datetime


class DriftDetector:
    """
    Monitors incoming data for distribution drift against training baseline.
    Uses PSI as primary drift signal — robust at large sample sizes.
    """
    
    def __init__(self, reference_data: pd.DataFrame, psi_threshold: float = 0.2):
        self.reference = reference_data
        self.psi_threshold = psi_threshold
        self.alerts = []
    
    def calculate_psi(self, expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
        """
        Population Stability Index.
        < 0.1:  no significant shift
        0.1-0.2: moderate shift, monitor
        > 0.2:  significant shift, retrain
        """
        breakpoints = np.linspace(
            min(expected.min(), actual.min()),
            max(expected.max(), actual.max()),
            bins + 1
        )
        
        expected_counts = np.histogram(expected, bins=breakpoints)[0] + 1
        actual_counts = np.histogram(actual, bins=breakpoints)[0] + 1
        
        expected_pct = expected_counts / expected_counts.sum()
        actual_pct = actual_counts / actual_counts.sum()
        
        psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
        return float(psi)
    
    def check_drift(self, current_data: pd.DataFrame) -> dict:
        """Run drift analysis on incoming data vs reference baseline."""
        results = {
            "timestamp": datetime.now().isoformat(),
            "samples_checked": len(current_data),
            "features_drifted": [],
            "overall_status": "healthy",
            "details": {}
        }
        
        for col in self.reference.columns:
            if col not in current_data.columns:
                continue
            
            ref_values = self.reference[col].dropna().values
            cur_values = current_data[col].dropna().values
            
            if len(cur_values) < 10:
                continue
            
            psi = self.calculate_psi(ref_values, cur_values)
            
            ref_mean = ref_values.mean()
            cur_mean = cur_values.mean()
            mean_shift = abs(cur_mean - ref_mean) / (ref_values.std() + 1e-6)
            
            is_drifted = psi > self.psi_threshold
            
            results["details"][col] = {
                "psi": round(psi, 4),
                "mean_shift_std": round(mean_shift, 4),
                "ref_mean": round(ref_mean, 4),
                "cur_mean": round(cur_mean, 4),
                "drifted": is_drifted
            }
            
            if is_drifted:
                results["features_drifted"].append(col)
        
        drift_count = len(results["features_drifted"])
        total_features = len(results["details"])
        drift_pct = drift_count / total_features if total_features > 0 else 0
        
        if drift_pct > 0.3:
            results["overall_status"] = "critical"
        elif drift_pct > 0.1:
            results["overall_status"] = "warning"
        
        if results["overall_status"] != "healthy":
            self.alerts.append({
                "timestamp": results["timestamp"],
                "status": results["overall_status"],
                "drifted_features": drift_count,
                "total_features": total_features
            })
        
        return results
    
    def get_alerts(self) -> list:
        return self.alerts

Overwriting pipeline/monitoring/drift_detector.py


In [4]:
from importlib import reload
import pipeline.monitoring.drift_detector as dm
reload(dm)
from pipeline.monitoring.drift_detector import DriftDetector

# Same setup
detector = DriftDetector(reference)

# TEST 1: Real production data — should be healthy now
result = detector.check_drift(production)
print("TEST 1: Real production data vs training reference")
print(f"  Status: {result['overall_status']}")
print(f"  Features drifted: {len(result['features_drifted'])} / {len(result['details'])}")
if result['features_drifted']:
    for f in result['features_drifted']:
        d = result['details'][f]
        print(f"    {f}: PSI={d['psi']}")

# TEST 2: Simulated drift
drifted = production.copy()
drifted['V14'] = drifted['V14'] + 2.0
drifted['V17'] = drifted['V17'] * 1.5
drifted['Amount'] = drifted['Amount'] * 3.0

result2 = detector.check_drift(drifted)
print(f"\nTEST 2: Simulated drift (V14 shifted, V17 scaled, Amount tripled)")
print(f"  Status: {result2['overall_status']}")
print(f"  Features drifted: {len(result2['features_drifted'])} / {len(result2['details'])}")
for f in result2['features_drifted']:
    d = result2['details'][f]
    print(f"    {f}: PSI={d['psi']}, mean shift: {d['ref_mean']:.3f} → {d['cur_mean']:.3f}")

print(f"\nAlerts: {len(detector.get_alerts())}")
for a in detector.get_alerts():
    print(f"  [{a['status'].upper()}] {a['drifted_features']}/{a['total_features']} features drifted")

TEST 1: Real production data vs training reference
  Status: healthy
  Features drifted: 2 / 40
    hour: PSI=6.0399
    is_night: PSI=0.9252

TEST 2: Simulated drift (V14 shifted, V17 scaled, Amount tripled)
  Status: healthy
  Features drifted: 4 / 40
    V14: PSI=2.5465, mean shift: 0.033 → 1.869
    V17: PSI=0.2377, mean shift: 0.015 → -0.091
    hour: PSI=6.0399, mean shift: 13.207 → 19.860
    is_night: PSI=0.9252, mean shift: 0.105 → 0.000

Alerts: 0


In [5]:
%%writefile pipeline/monitoring/drift_detector.py
"""
Drift Detection Service
Enterprise MLOps Platform
"""

import numpy as np
import pandas as pd
from datetime import datetime


class DriftDetector:
    """
    Monitors incoming data for distribution drift against training baseline.
    Uses PSI as primary signal. Supports critical feature watchlist.
    """
    
    def __init__(self, reference_data: pd.DataFrame, psi_threshold: float = 0.2,
                 critical_features: list = None):
        self.reference = reference_data
        self.psi_threshold = psi_threshold
        self.critical_features = critical_features or []
        self.alerts = []
    
    def calculate_psi(self, expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
        breakpoints = np.linspace(
            min(expected.min(), actual.min()),
            max(expected.max(), actual.max()),
            bins + 1
        )
        
        expected_counts = np.histogram(expected, bins=breakpoints)[0] + 1
        actual_counts = np.histogram(actual, bins=breakpoints)[0] + 1
        
        expected_pct = expected_counts / expected_counts.sum()
        actual_pct = actual_counts / actual_counts.sum()
        
        psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
        return float(psi)
    
    def check_drift(self, current_data: pd.DataFrame, exclude_features: list = None) -> dict:
        exclude = exclude_features or []
        
        results = {
            "timestamp": datetime.now().isoformat(),
            "samples_checked": len(current_data),
            "features_drifted": [],
            "critical_features_drifted": [],
            "overall_status": "healthy",
            "details": {}
        }
        
        for col in self.reference.columns:
            if col not in current_data.columns or col in exclude:
                continue
            
            ref_values = self.reference[col].dropna().values
            cur_values = current_data[col].dropna().values
            
            if len(cur_values) < 10:
                continue
            
            psi = self.calculate_psi(ref_values, cur_values)
            
            ref_mean = ref_values.mean()
            cur_mean = cur_values.mean()
            mean_shift = abs(cur_mean - ref_mean) / (ref_values.std() + 1e-6)
            
            is_drifted = psi > self.psi_threshold
            
            results["details"][col] = {
                "psi": round(psi, 4),
                "mean_shift_std": round(mean_shift, 4),
                "ref_mean": round(ref_mean, 4),
                "cur_mean": round(cur_mean, 4),
                "drifted": is_drifted,
                "is_critical": col in self.critical_features
            }
            
            if is_drifted:
                results["features_drifted"].append(col)
                if col in self.critical_features:
                    results["critical_features_drifted"].append(col)
        
        # Determine status
        drift_count = len(results["features_drifted"])
        total_features = len(results["details"])
        drift_pct = drift_count / total_features if total_features > 0 else 0
        
        if results["critical_features_drifted"]:
            results["overall_status"] = "critical"
        elif drift_pct > 0.3:
            results["overall_status"] = "critical"
        elif drift_pct > 0.1:
            results["overall_status"] = "warning"
        
        if results["overall_status"] != "healthy":
            self.alerts.append({
                "timestamp": results["timestamp"],
                "status": results["overall_status"],
                "drifted_features": drift_count,
                "critical_drifted": results["critical_features_drifted"],
                "total_features": total_features,
                "recommendation": "retrain" if results["overall_status"] == "critical" else "monitor"
            })
        
        return results
    
    def get_alerts(self) -> list:
        return self.alerts
    
    def should_retrain(self) -> bool:
        if not self.alerts:
            return False
        return self.alerts[-1]["status"] == "critical"

Overwriting pipeline/monitoring/drift_detector.py


In [6]:
from importlib import reload
import pipeline.monitoring.drift_detector as dm
reload(dm)
from pipeline.monitoring.drift_detector import DriftDetector

# Top features from our model — these are the ones that matter most
critical_features = ['V14', 'V17', 'V12', 'V10', 'fraud_risk_signal', 'v14_x_v12']

# Exclude time features — they drift due to dataset structure, not real drift
exclude = ['hour', 'is_night']

detector = DriftDetector(reference, critical_features=critical_features)

# TEST 1: Real production data
result = detector.check_drift(production, exclude_features=exclude)
print("TEST 1: Real production data")
print(f"  Status: {result['overall_status']}")
print(f"  Features drifted: {len(result['features_drifted'])} / {len(result['details'])}")
print(f"  Critical features drifted: {result['critical_features_drifted']}")
print(f"  Should retrain: {detector.should_retrain()}")

# TEST 2: Simulated drift on key fraud features
drifted = production.copy()
drifted['V14'] = drifted['V14'] + 2.0
drifted['V17'] = drifted['V17'] * 1.5
drifted['Amount'] = drifted['Amount'] * 3.0

result2 = detector.check_drift(drifted, exclude_features=exclude)
print(f"\nTEST 2: Simulated drift on fraud-critical features")
print(f"  Status: {result2['overall_status']}")
print(f"  Features drifted: {len(result2['features_drifted'])} / {len(result2['details'])}")
print(f"  Critical features drifted: {result2['critical_features_drifted']}")
print(f"  Should retrain: {detector.should_retrain()}")

for f in result2['features_drifted']:
    d = result2['details'][f]
    crit = " *** CRITICAL ***" if d['is_critical'] else ""
    print(f"    {f}: PSI={d['psi']}, mean {d['ref_mean']:.3f} → {d['cur_mean']:.3f}{crit}")

print(f"\nAlerts: {len(detector.get_alerts())}")
for a in detector.get_alerts():
    print(f"  [{a['status'].upper()}] {a['recommendation']} — critical drifted: {a['critical_drifted']}")

TEST 1: Real production data
  Status: healthy
  Features drifted: 0 / 38
  Critical features drifted: []
  Should retrain: False

TEST 2: Simulated drift on fraud-critical features
  Status: critical
  Features drifted: 2 / 38
  Critical features drifted: ['V14', 'V17']
  Should retrain: True
    V14: PSI=2.5465, mean 0.033 → 1.869 *** CRITICAL ***
    V17: PSI=0.2377, mean 0.015 → -0.091 *** CRITICAL ***

Alerts: 1
  [CRITICAL] retrain — critical drifted: ['V14', 'V17']
